<a href="https://colab.research.google.com/github/AashikML/The-Data-Refinery/blob/main/Data_Refinery_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import sqlite3
import pandas as pd

# ==========================================
# STEP 1 & 2: CONNECT, CREATE SCHEMA & INSERT DATA
# ==========================================
conn = sqlite3.connect('ecommerce.db')
cursor = conn.cursor()

# Tables creation with Primary and Foreign Keys
cursor.executescript('''
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    amount REAL,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

-- Insert Raw Data into Customers
INSERT INTO customers VALUES
(1, 'Aashik', 'Delhi'),
(2, 'Rahul', 'Mumbai'),
(3, 'Ismarika', 'Bangalore'),
(4, 'Vikram', 'Delhi');

-- Insert Raw Data into Orders
INSERT INTO orders VALUES
(101, 1, 15000.00, 'Completed'),
(102, 1, 8000.00, 'Completed'),
(103, 2, 5000.00, 'Cancelled'),
(104, 3, 22000.00, 'Completed'),
(105, 3, 12000.00, 'Completed'),
(106, 4, 3000.00, 'Pending');
''')

conn.commit()
print("✅ Step 1 & 2: Database, Tables & Data Setup Done!")

# ==========================================
# STEP 3: DATA CLEANING (UPDATE & DELETE)
# ==========================================
cursor.execute("UPDATE orders SET status = 'Completed' WHERE status = 'Pending';")
cursor.execute("DELETE FROM orders WHERE status = 'Cancelled';")
conn.commit()
print("✅ Step 3: Data Cleaning (UPDATE & DELETE) Done!")

# ==========================================
# STEP 4 & 5: AGGREGATION QUERY & PANDAS LOAD
# ==========================================
pipeline_query = '''
SELECT
    c.name AS customer_name,
    c.city,
    SUM(o.amount) AS total_spent,
    COUNT(o.order_id) AS order_count
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'Completed'
GROUP BY c.customer_id, c.name, c.city
HAVING SUM(o.amount) > 20000;
'''

# Load into Pandas DataFrame
df_high_value_customers = pd.read_sql_query(pipeline_query, conn)

# Close Connection safely
conn.close()

# Print Final Output
print("\n🔥 --- THE DATA REFINERY PIPELINE OUTPUT --- 🔥")
print(df_high_value_customers)

✅ Step 1 & 2: Database, Tables & Data Setup Done!
✅ Step 3: Data Cleaning (UPDATE & DELETE) Done!

🔥 --- THE DATA REFINERY PIPELINE OUTPUT --- 🔥
  customer_name       city  total_spent  order_count
0        Aashik      Delhi      23000.0            2
1      Ismarika  Bangalore      34000.0            2
